# Stock Market Return Statistical Analyzer
**SI Lab Project — Ramdeobaba University, Nagpur**

Historical stock-return analysis using data collection, descriptive statistics, visualization, sampling, hypothesis testing, correlation, regression and classification.

**Data source:** Yahoo Finance via `yfinance`; S&P 500 ticker list via Wikipedia.

## 1. Import Libraries and Setup

In [ ]:
%pip -q install yfinance requests beautifulsoup4 lxml

In [ ]:
import os
import re
import sys
import pickle
import warnings
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from bs4 import BeautifulSoup
from scipy import stats

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("stock_analyzer_data")
MODEL_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Project directories created.")

## 2. Project Configuration

In [ ]:
TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL"]
BENCHMARK = "^GSPC"
PERIOD = "5y"
INTERVAL = "1d"

print("Stocks:", TICKERS)
print("Benchmark:", BENCHMARK)
print("Period:", PERIOD)
print("Interval:", INTERVAL)

## 3. Web Scraping: S&P 500 Ticker List
Scrape the S&P 500 constituents table using Requests and BeautifulSoup.

In [ ]:
wiki_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (compatible; SI-Lab-Student-Project/1.0)"}

sp500_tickers = []
try:
    response = requests.get(wiki_url, headers=headers, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", {"id": "constituents"})
    if table is not None:
        for row in table.find_all("tr")[1:]:
            cols = row.find_all("td")
            if cols:
                symbol = cols[0].get_text(" ", strip=True).replace(".", "-")
                sp500_tickers.append(symbol)
    print("Scraped ticker count:", len(sp500_tickers))
    print("First 10 scraped tickers:", sp500_tickers[:10])
except Exception as e:
    print("Ticker scraping was unavailable:", str(e))
    print("Continuing with the configured tickers.")

if sp500_tickers:
    pd.DataFrame({"Ticker": sp500_tickers}).to_csv(DATA_DIR / "sp500_ticker_list.csv", index=False)
    print("Saved ticker list to:", DATA_DIR / "sp500_ticker_list.csv")

## 4. Download Historical Market Data

In [ ]:
all_tickers = TICKERS + [BENCHMARK]
raw = yf.download(
    tickers=all_tickers,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    group_by="ticker",
    progress=False,
    threads=True
)

if raw is None or raw.empty:
    raise RuntimeError(
        "No market data was returned. Check internet access, ticker symbols, "
        "or try again later."
    )

print("Downloaded data shape:", raw.shape)
display(raw.head())

In [ ]:
price_data = {}

if isinstance(raw.columns, pd.MultiIndex):
    first_level = set(raw.columns.get_level_values(0))
    for ticker in all_tickers:
        if ticker in first_level:
            temp = raw[ticker].copy()
        else:
            try:
                temp = raw.xs(ticker, axis=1, level=1).copy()
            except Exception:
                continue
        if "Close" in temp.columns:
            temp = temp.reset_index()
            temp["Date"] = pd.to_datetime(temp["Date"]).dt.tz_localize(None)
            temp = temp.sort_values("Date").drop_duplicates(subset=["Date"])
            price_data[ticker] = temp
else:
    # Single-ticker fallback
    temp = raw.reset_index()
    temp["Date"] = pd.to_datetime(temp["Date"]).dt.tz_localize(None)
    price_data[TICKERS[0]] = temp

available_tickers = list(price_data.keys())
print("Tickers with usable data:", available_tickers)

if not any(t in price_data for t in TICKERS):
    raise RuntimeError("No configured stock ticker has a usable Close price series.")

## 5. Data Inspection and Cleaning

In [ ]:
close_prices = pd.DataFrame()
for ticker, temp in price_data.items():
    if "Close" in temp.columns:
        series = temp.set_index("Date")["Close"].rename(ticker)
        close_prices = pd.concat([close_prices, series], axis=1)

close_prices = close_prices.sort_index()
close_prices = close_prices[~close_prices.index.duplicated(keep="last")]

print("First 5 rows:")
display(close_prices.head())
print("Last 5 rows:")
display(close_prices.tail())
print("Number of rows and columns:", close_prices.shape)
print("Column names:", close_prices.columns.tolist())
print("\nData types:")
print(close_prices.dtypes)
print("\nMissing values:")
display(close_prices.isnull().sum())
print("\nDuplicate dates:", close_prices.index.duplicated().sum())

In [ ]:
close_prices = close_prices.dropna(axis=1, how="all")
close_prices = close_prices.drop_duplicates()
close_prices.to_csv(DATA_DIR / "stock_close_prices.csv", index_label="Date")

print("Cleaned price data shape:", close_prices.shape)
print("Saved:", DATA_DIR / "stock_close_prices.csv")

## 6. Calculate Daily Returns
Daily percentage return is calculated from adjusted closing prices.

In [ ]:
returns = close_prices.pct_change(fill_method=None) * 100
returns = returns.replace([np.inf, -np.inf], np.nan)

print("First 5 return rows:")
display(returns.head())
print("\nMissing return values:")
display(returns.isnull().sum())

In [ ]:
returns.reset_index().to_csv(DATA_DIR / "daily_stock_returns.csv", index=False)
print("Saved daily returns to:", DATA_DIR / "daily_stock_returns.csv")

stock_tickers_available = [t for t in TICKERS if t in returns.columns]
if not stock_tickers_available:
    stock_tickers_available = [c for c in returns.columns if c != BENCHMARK]
if not stock_tickers_available:
    raise RuntimeError("No stock return series available.")
PRIMARY_TICKER = stock_tickers_available[0]
stock_returns = returns[PRIMARY_TICKER].dropna()

print("Primary stock for single-series practicals:", PRIMARY_TICKER)
print("Valid daily return observations:", len(stock_returns))

## 7. Descriptive Statistical Analysis

In [ ]:
def descriptive_statistics(series):
    series = pd.Series(series).dropna()
    q1 = series.quantile(0.25)
    q2 = series.quantile(0.50)
    q3 = series.quantile(0.75)
    result = {
        "Count": series.count(),
        "Mean": series.mean(),
        "Median": series.median(),
        "Mode": series.mode().iloc[0] if not series.mode().empty else np.nan,
        "Variance": series.var(),
        "Standard Deviation": series.std(),
        "Minimum": series.min(),
        "Q1": q1,
        "Q2": q2,
        "Q3": q3,
        "Maximum": series.max(),
        "IQR": q3 - q1
    }
    return pd.Series(result)

stats_table = pd.DataFrame({
    ticker: descriptive_statistics(returns[ticker])
    for ticker in stock_tickers_available
}).T

display(stats_table.round(6))
print("\nPandas summary statistics:")
display(returns[stock_tickers_available].describe().T)

## 8. Statistical Visualizations
Histogram, box plot, bar chart, pie chart, scatter plot and line chart.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(stock_returns, bins=40, edgecolor="black")
plt.title(f"Distribution of Daily Returns — {PRIMARY_TICKER}")
plt.xlabel("Daily return (%)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(x=stock_returns)
plt.title(f"Box Plot of Daily Returns — {PRIMARY_TICKER}")
plt.xlabel("Daily return (%)")
plt.tight_layout()
plt.show()

mean_returns = returns[stock_tickers_available].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
mean_returns.plot(kind="bar")
plt.title("Mean Daily Return by Stock")
plt.xlabel("Ticker")
plt.ylabel("Mean daily return (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

absolute_movement = returns[stock_tickers_available].abs().mean().sort_values(ascending=False)
plt.figure(figsize=(8, 8))
plt.pie(absolute_movement, labels=absolute_movement.index, autopct="%1.1f%%", startangle=90)
plt.title("Share of Average Absolute Daily Return by Stock")
plt.tight_layout()
plt.show()

if BENCHMARK in returns.columns and PRIMARY_TICKER in returns.columns:
    paired = returns[[PRIMARY_TICKER, BENCHMARK]].dropna()
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=paired, x=BENCHMARK, y=PRIMARY_TICKER, alpha=0.5)
    plt.title(f"{PRIMARY_TICKER} Return vs S&P 500 Return")
    plt.xlabel("S&P 500 daily return (%)")
    plt.ylabel(f"{PRIMARY_TICKER} daily return (%)")
    plt.tight_layout()
    plt.show()

growth = (1 + returns[stock_tickers_available] / 100).cumprod()
plt.figure(figsize=(12, 6))
for ticker in growth.columns:
    plt.plot(growth.index, growth[ticker], label=ticker)
plt.title("Cumulative Growth of $1 from Daily Returns")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Sampling Methods

In [ ]:
sample_size = min(30, len(stock_returns))
simple_random_sample = stock_returns.sample(n=sample_size, random_state=42)
print("Simple random sample:")
display(simple_random_sample.head(10))
print("Sample mean:", simple_random_sample.mean())
print("Full-series mean:", stock_returns.mean())

k = max(1, len(stock_returns) // max(sample_size, 1))
systematic_sample = stock_returns.iloc[::k].head(sample_size)
print("\nSystematic sample size:", len(systematic_sample))
print("Systematic sample mean:", systematic_sample.mean())

returns_frame = stock_returns.rename("Return").to_frame()
returns_frame["Year"] = returns_frame.index.year
stratified_sample = (
    returns_frame.groupby("Year", group_keys=False)
    .apply(lambda group: group.sample(
        n=max(1, min(len(group), int(np.ceil(sample_size / returns_frame["Year"].nunique())))),
        random_state=42
    ))
)
print("\nStratified sample by year:")
display(stratified_sample.head(10))

## 10. Sampling Distribution and Central Limit Theorem

In [ ]:
population = stock_returns.to_numpy()
population_mean = np.mean(population)
population_std = np.std(population, ddof=1)

sample_sizes = [5, 30, 100]
number_of_samples = 1000

for n in sample_sizes:
    if len(population) < n:
        continue
    sample_means = []
    for i in range(number_of_samples):
        sample = np.random.choice(population, size=n, replace=True)
        sample_means.append(np.mean(sample))
    sample_means = np.array(sample_means)

    plt.figure(figsize=(8, 5))
    plt.hist(sample_means, bins=30, edgecolor="black")
    plt.axvline(population_mean, linestyle="--", label="Population mean")
    plt.title(f"Sampling Distribution of Mean Returns (n={n})")
    plt.xlabel("Sample mean daily return (%)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Sample size:", n)
    print("Mean of sample means:", np.mean(sample_means))
    print("Theoretical standard error:", population_std / np.sqrt(n))
    print("Observed standard error:", np.std(sample_means, ddof=1))
    print()

## 11. Point Estimation and Confidence Interval

In [ ]:
sample_mean = stock_returns.mean()
sample_std = stock_returns.std(ddof=1)
n = stock_returns.count()
standard_error = sample_std / np.sqrt(n)
confidence_level = 0.95
alpha = 1 - confidence_level
critical_t = stats.t.ppf(1 - alpha / 2, df=n - 1)
ci_lower = sample_mean - critical_t * standard_error
ci_upper = sample_mean + critical_t * standard_error

print("Point estimate of mean return:", sample_mean)
print("Standard error:", standard_error)
print("95% confidence interval:", (ci_lower, ci_upper))

## 12. Z-Test and One-Sample T-Test

In [ ]:
# H0: population mean return = 0
# H1: population mean return != 0
t_stat, t_p_value = stats.ttest_1samp(stock_returns, popmean=0.0)
print("One-sample T-test")
print("T statistic:", t_stat)
print("P-value:", t_p_value)
print("Decision at 5% significance:", "Reject H0" if t_p_value < 0.05 else "Fail to reject H0")

z_stat = (sample_mean - 0.0) / standard_error
z_p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print("\nLarge-sample Z approximation")
print("Z statistic:", z_stat)
print("Approximate two-sided p-value:", z_p_value)
print("Interpret the Z approximation cautiously because population SD is estimated.")

## 13. Two-Sample and Paired T-Tests

In [ ]:
if len(stock_tickers_available) >= 2:
    ticker_a, ticker_b = stock_tickers_available[:2]
    a = returns[ticker_a].dropna()
    b = returns[ticker_b].dropna()
    independent_result = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    print("Welch Two-Sample T-Test:", ticker_a, "vs", ticker_b)
    print("T statistic:", independent_result.statistic)
    print("P-value:", independent_result.pvalue)

if len(stock_tickers_available) >= 2:
    paired_data = returns[[ticker_a, ticker_b]].dropna()
    paired_result = stats.ttest_rel(paired_data[ticker_a], paired_data[ticker_b])
    print("\nPaired T-Test on matching dates:", ticker_a, "vs", ticker_b)
    print("Paired observations:", len(paired_data))
    print("T statistic:", paired_result.statistic)
    print("P-value:", paired_result.pvalue)

## 14. Correlation and Covariance

In [ ]:
correlation_pearson = returns[stock_tickers_available].corr(method="pearson")
correlation_spearman = returns[stock_tickers_available].corr(method="spearman")
covariance_matrix = returns[stock_tickers_available].cov()

print("Pearson Correlation Matrix:")
display(correlation_pearson.round(4))
print("Spearman Correlation Matrix:")
display(correlation_spearman.round(4))
print("Covariance Matrix:")
display(covariance_matrix.round(6))

plt.figure(figsize=(9, 7))
sns.heatmap(correlation_pearson, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Pearson Correlation Matrix of Daily Returns")
plt.tight_layout()
plt.show()

In [ ]:
if BENCHMARK in returns.columns:
    corr_data = returns[[PRIMARY_TICKER, BENCHMARK]].dropna()
    if len(corr_data) >= 3:
        pearson_r, pearson_p = stats.pearsonr(corr_data[PRIMARY_TICKER], corr_data[BENCHMARK])
        spearman_r, spearman_p = stats.spearmanr(corr_data[PRIMARY_TICKER], corr_data[BENCHMARK])
        print("Pearson r:", pearson_r, "p-value:", pearson_p)
        print("Spearman rho:", spearman_r, "p-value:", spearman_p)

## 15. Chi-Square Tests

In [ ]:
direction = np.where(stock_returns > 0, "Positive", "Non-positive")
observed_counts = pd.Series(direction).value_counts().reindex(
    ["Positive", "Non-positive"], fill_value=0
)
expected_counts = np.array([observed_counts.sum() / 2, observed_counts.sum() / 2])

chi2_stat, chi2_p = stats.chisquare(
    f_obs=observed_counts.to_numpy(),
    f_exp=expected_counts
)
print("Observed counts:")
display(observed_counts)
print("Expected counts:", expected_counts)
print("Chi-square statistic:", chi2_stat)
print("P-value:", chi2_p)

In [ ]:
weekday_frame = stock_returns.rename("Return").to_frame()
weekday_frame["Weekday"] = weekday_frame.index.day_name()
weekday_frame["Direction"] = np.where(weekday_frame["Return"] > 0, "Positive", "Non-positive")
contingency_table = pd.crosstab(weekday_frame["Weekday"], weekday_frame["Direction"])
print("Contingency table:")
display(contingency_table)

if contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
    chi2_ind, p_ind, dof_ind, expected_ind = stats.chi2_contingency(contingency_table)
    print("Chi-square statistic:", chi2_ind)
    print("Degrees of freedom:", dof_ind)
    print("P-value:", p_ind)
    print("Expected frequencies:")
    display(pd.DataFrame(expected_ind, index=contingency_table.index, columns=contingency_table.columns))

## 16. Feature Engineering for Predictive Models
Features use past observations to predict the next trading day's return and direction.

In [ ]:
model_data = pd.DataFrame(index=returns.index)
model_data["Return"] = returns[PRIMARY_TICKER]
model_data["Return_Lag1"] = model_data["Return"].shift(1)
model_data["Return_Lag2"] = model_data["Return"].shift(2)
model_data["Return_Lag5"] = model_data["Return"].shift(5)
model_data["Volatility_5D"] = model_data["Return"].rolling(5).std()
model_data["Volatility_20D"] = model_data["Return"].rolling(20).std()
model_data["Moving_Average_5D"] = model_data["Return"].rolling(5).mean()
model_data["Moving_Average_20D"] = model_data["Return"].rolling(20).mean()

if BENCHMARK in returns.columns:
    model_data["Benchmark_Return"] = returns[BENCHMARK]

model_data["Target_Return"] = model_data["Return"].shift(-1)
model_data["Target_Direction"] = (model_data["Target_Return"] > 0).astype(int)
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna()

feature_cols = [
    c for c in model_data.columns
    if c not in ["Return", "Target_Return", "Target_Direction"]
]
X = model_data[feature_cols]
y_reg = model_data["Target_Return"]
y_clf = model_data["Target_Direction"]

print("Model data shape:", model_data.shape)
print("Features:", feature_cols)
print("Regression target: next-day return")
print("Classification target: next-day return > 0")
display(model_data.head())

## 17. Chronological Train/Test Split
The latest 20% of observations are reserved for testing; data is not shuffled.

In [ ]:
split_index = int(len(model_data) * 0.8)
split_index = max(1, min(split_index, len(model_data) - 1))

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_reg_train = y_reg.iloc[:split_index].copy()
y_reg_test = y_reg.iloc[split_index:].copy()
y_clf_train = y_clf.iloc[:split_index].copy()
y_clf_test = y_clf.iloc[split_index:].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("First test date:", X_test.index.min())
print("Last test date:", X_test.index.max())

## 18. Train and Compare Regression Models

In [ ]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1, max_iter=10000),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
    )
}

trained_regression_models = {}
regression_results = []

for name, estimator in regression_models.items():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", estimator)
    ])
    model.fit(X_train, y_reg_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_reg_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_reg_test, predictions))
    r2 = r2_score(y_reg_test, predictions)

    trained_regression_models[name] = model
    regression_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

regression_results_df = pd.DataFrame(regression_results).sort_values("RMSE")
print("Regression Model Comparison:")
display(regression_results_df)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=regression_results_df, x="Model", y="RMSE")
plt.xticks(rotation=30, ha="right")
plt.title("Regression Model Comparison (Lower RMSE is Better)")
plt.tight_layout()
plt.show()

best_regression_name = regression_results_df.iloc[0]["Model"]
best_regression_model = trained_regression_models[best_regression_name]
print("Best regression model by test RMSE:", best_regression_name)
print("Note: final model selection should use validation data; test results are reported as held-out evaluation.")

## 19. Train and Compare Classification Models

In [ ]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=8, random_state=42,
        n_jobs=-1, class_weight="balanced"
    )
}

trained_classification_models = {}
classification_results = []

for name, estimator in classification_models.items():
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", estimator)
    ])
    model.fit(X_train, y_clf_train)
    predictions = model.predict(X_test)

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_clf_test, predictions),
        "Precision": precision_score(y_clf_test, predictions, zero_division=0),
        "Recall": recall_score(y_clf_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_clf_test, predictions, zero_division=0)
    })
    trained_classification_models[name] = model

classification_results_df = pd.DataFrame(classification_results).sort_values(
    "F1-Score", ascending=False
)
print("Classification Model Comparison:")
display(classification_results_df)

In [ ]:
best_classification_name = classification_results_df.iloc[0]["Model"]
best_classification_model = trained_classification_models[best_classification_name]
classification_predictions = best_classification_model.predict(X_test)

print("Selected classification model by F1-score:", best_classification_name)
print("\nClassification report:")
print(classification_report(y_clf_test, classification_predictions, zero_division=0))

cm = confusion_matrix(y_clf_test, classification_predictions, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-positive", "Positive"],
            yticklabels=["Non-positive", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 20. Save Models and Datasets

In [ ]:
regression_model_path = MODEL_DIR / "best_return_regression_model.pkl"
classification_model_path = MODEL_DIR / "best_direction_classification_model.pkl"

with open(regression_model_path, "wb") as file:
    pickle.dump(best_regression_model, file)

with open(classification_model_path, "wb") as file:
    pickle.dump(best_classification_model, file)

regression_results_df.to_csv(DATA_DIR / "regression_model_comparison.csv", index=False)
classification_results_df.to_csv(DATA_DIR / "classification_model_comparison.csv", index=False)
model_data.to_csv(DATA_DIR / "modeling_dataset.csv", index_label="Date")

print("Saved regression model:", regression_model_path)
print("Saved classification model:", classification_model_path)
print("Saved modeling dataset and comparison tables.")

## 21. Example Prediction

In [ ]:
latest_features = X.iloc[[-1]]
predicted_return = best_regression_model.predict(latest_features)[0]
predicted_direction = best_classification_model.predict(latest_features)[0]

print("Ticker:", PRIMARY_TICKER)
print("Feature date:", latest_features.index[0])
print("Estimated next-day return (%):", round(float(predicted_return), 4))
print("Predicted next-day direction:", "Positive" if predicted_direction == 1 else "Non-positive")
print("This is a model output for demonstration, not a reliable trading signal.")

## 22. Findings and Limitations

After running the notebook, discuss actual results:
- Return distribution, volatility and outliers.
- Confidence intervals and hypothesis-test results.
- Stock/benchmark correlations.
- Regression and classification metrics.
- Baseline comparisons and data limitations.

Statistical significance does not establish profitability or causation.

## 23. Deliverables

Saved files are stored in `stock_analyzer_data/` and `models/`. Export the notebook as PDF after running and checking all cells.